In [2]:
print("Ok")

Ok


In [4]:
%pwd

'd:\\Programming\\GenAI\\Medical_Chatbot\\research'

In [5]:
import os
os.chdir("../") #to go to data folder we are coming out of the research folder

In [6]:
%pwd

'd:\\Programming\\GenAI\\Medical_Chatbot'

In [7]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Praveena Varshini M\AppData\Local\Temp\ipykernel_13464\3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [8]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents


In [9]:
extracted_data=load_pdf_file(data='Data/')

In [ ]:
extracted_data

In [10]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [13]:
# text_chunks

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [14]:
from langchain_openai import OpenAIEmbeddings

def download_openai_embeddings():
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        dimensions=512
    )
    return embeddings

In [15]:
embeddings = download_openai_embeddings()

In [16]:
query_result = embeddings.embed_query("Hello world")

print("Length:", len(query_result))

Length: 512


In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
print(PINECONE_API_KEY is not None)

True


In [18]:
from pinecone import Pinecone

pc = Pinecone(PINECONE_API_KEY)

index_name = "medibot"

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
) 

In [ ]:
vectorstore.add_documents(text_chunks)
#pinecone use the above index and openai embedding and store it in db

In [27]:
index.describe_index_stats()

{'dimension': 512,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5860}},
 'total_vector_count': 5860,
 'vector_type': 'dense'}

In [28]:
vectorstore

In [29]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [30]:
retrieved_docs = retriever.invoke("What is Acne?")

In [31]:
retrieved_docs

[Document(id='b677bfcb-a21b-486b-bf4e-2985072c086c', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 239.0, 'page_label': '240', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='Isotretinoin (Accutane) is prescribed only for very\nsevere, disfiguring acne.\nAcne is a skin condition that occurs when pores or\nhair follicles become blocked. This allows a waxy\nmaterial, sebum, to collect inside the pores or follicles.\nNormally, sebum flows out onto the skin and hair to\nform a protective coating, but when it cannot get out,\nsmall swellings develop on the skin surface. Bacteria\nand dead skin cells can also collect that can cause\ninflammation. Swellings that are small and not'),
 Document(id='bec9b247-4fa3-41dc-a088-6faf0ed57ca5', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page

In [40]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [36]:
llm = ChatOpenAI(
    temperature=0.4,
    max_tokens=500,
    model="gpt-4o-mini"
)

In [37]:
# System prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "Context:\n{context}"
)


# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [44]:
rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [46]:
# Question 1
response = rag_chain.invoke(
     "What is Acromegaly and gigantism?"
)

print(response)

Acromegaly is a disorder characterized by the abnormal release of growth hormone from the pituitary gland, leading to increased growth in bones and soft tissues, along with various other bodily disturbances. Gigantism, which occurs in children before the closure of growth plates, results in excessive height due to similar hormonal imbalances. Both conditions stem from the overproduction of growth hormone, often caused by tumors in the pituitary gland or other glands that produce growth hormone-releasing hormone (GHRH).
